# 06.5 — GroupBy and Aggregation

## Learning Objectives

By the end of this notebook you should be able to:

- Explain the **split–apply–combine** pattern
- Use `.groupby()` with a single key and multiple keys
- Apply aggregation functions: `.mean()`, `.sum()`, `.count()`, `.min()`, `.max()`
- Use `.agg()` to compute multiple aggregations at once
- Sort grouped results and reshape with `.reset_index()`

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(df.shape)
df.head()

## 1. The Split–Apply–Combine Pattern

**GroupBy** is how pandas implements a fundamental pattern called **split–apply–combine**:

1. **Split** — divide the DataFrame into groups based on one or more column values.
2. **Apply** — compute something within each group (a sum, mean, count, custom function…).
3. **Combine** — assemble the per-group results into a single output.

```
DataFrame
┌─────────────────────────────────┐
│ sex    survived   fare          │
│ male        0     7.25          │
│ female      1    71.28          │
│ female      1     7.92          │
│ male        0    53.10          │
└─────────────────────────────────┘
         │
      SPLIT by sex
         │
┌──────────────────┐   ┌──────────────────────┐
│ GROUP: male      │   │ GROUP: female        │
│ survived  fare   │   │ survived    fare     │
│     0     7.25   │   │     1      71.28     │
│     0    53.10   │   │     1       7.92     │
└──────────────────┘   └──────────────────────┘
         │                      │
      APPLY .mean()          APPLY .mean()
         │                      │
       0.19                    0.74
         │                      │
         └────────COMBINE───────┘
                     │
          sex    survived
          female   0.74
          male     0.19
```

## 2. Basic GroupBy

Call `.groupby("column")` and chain an aggregation. The classic Titanic question: how did survival rates differ by sex?

In [ ]:
# .mean() on a 0/1 column gives the proportion of 1s — i.e., the survival rate
df.groupby("sex")["survived"].mean()

In [ ]:
# Survival rate by passenger class
df.groupby("pclass")["survived"].mean()

In [ ]:
# Average fare by class
df.groupby("pclass")["fare"].mean().round(2)

## 3. Multiple Aggregation Functions

In [ ]:
# Count passengers per class
df.groupby("pclass")["survived"].count()

In [ ]:
# .size() counts every row in each group (even if there were NaN values)
df.groupby("pclass").size()

## 4. Multiple Aggregations with `.agg()`

`.agg()` applies several aggregations in one call.

In [ ]:
# Summary statistics for fare, broken down by class
df.groupby("pclass")["fare"].agg(["mean", "median", "min", "max", "count"]).round(2)

In [ ]:
# Named aggregations: specify output column names explicitly
df.groupby("pclass").agg(
    passengers    = ("survived", "count"),
    survival_rate = ("survived", "mean"),
    avg_fare      = ("fare",     "mean"),
    avg_age       = ("age",      "mean"),
).round(2)

The named aggregation syntax `output_col=(source_col, function)` is clean and self-documenting.

## 5. Grouping by Multiple Keys

Pass a list to `.groupby()` to split by the combination of two or more columns.

In [ ]:
# Survival rate by class AND sex
df.groupby(["pclass", "sex"])["survived"].mean().round(2)

In [ ]:
# Full breakdown
df.groupby(["pclass", "sex"]).agg(
    count         = ("survived", "count"),
    survival_rate = ("survived", "mean"),
).round(2)

## 6. Sorting and Resetting the Index

After a groupby, the grouping columns become the **index** of the result. `.reset_index()` promotes them back to regular columns, which makes the result easier to work with.

In [ ]:
# Without reset_index: pclass is the index
result = df.groupby("pclass")["fare"].mean()
print(type(result))
print(result)

In [ ]:
# With reset_index: pclass is a regular column
result_df = df.groupby("pclass")["fare"].mean().reset_index()
result_df

In [ ]:
# Sort by survival rate, highest to lowest
df.groupby("pclass")["survived"].mean() \
  .reset_index() \
  .rename(columns={"survived": "survival_rate"}) \
  .sort_values("survival_rate", ascending=False)

## 7. A Complete Grouped Analysis

*For each passenger class and sex, how many passengers were there, and what fraction survived?*

In [ ]:
analysis = (
    df.groupby(["pclass", "sex"])
    .agg(
        passengers    = ("survived", "count"),
        survival_rate = ("survived", "mean"),
        avg_age       = ("age",      "mean"),
        avg_fare      = ("fare",     "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values(["pclass", "sex"])
)
analysis

In [ ]:
# Highlight the gender disparity
print("Survival rates by class and sex:")
print()
for pclass in sorted(df["pclass"].unique()):
    male   = df.loc[(df["pclass"]==pclass) & (df["sex"]=="male"),   "survived"].mean()
    female = df.loc[(df["pclass"]==pclass) & (df["sex"]=="female"), "survived"].mean()
    print(f"Class {pclass}:  male {male:.0%}  |  female {female:.0%}")

## Your Turn

1. What is the average age of survivors vs non-survivors?
2. What is the average fare paid by passengers who had family aboard (`sibsp > 0` or `parch > 0`) vs those traveling alone?
3. Build a summary table for each sex: count, average age, average fare, and survival rate. Sort by survival rate descending.
4. How many passengers in each class traveled with zero siblings/spouses (`sibsp == 0`) vs at least one?

In [ ]:
# Your code here